# Cross-Dataset Phishing Website Detection
## Notebook 5: Cross-Dataset Generalization Experiments (The Core Contribution)

**Central Research Question:**
> *"Do phishing detection models generalize across distinct datasets, or do they merely memorize dataset-specific artifacts?"*

**Experimental Design:**
- **Experiment 3:** Train on **PhiUSIIL Aligned** $\to$ Test on **UCI Phishing Aligned**.
- **Experiment 4:** Train on **UCI Phishing Aligned** $\to$ Test on **PhiUSIIL Aligned**.


In [ ]:
# Environment & Path Setup (supports local execution and Google Colab)
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/phishing-project')
except ImportError:
    BASE = Path('..').resolve() if Path('.').resolve().name == 'notebooks' else Path('.').resolve()

sys.path.insert(0, str(BASE))
print(f"Project base path: {BASE}")

from src.data_utils import project_paths
dirs = project_paths(BASE)
PROC_DIR = dirs['processed']
TAB_DIR = dirs['tables']
MOD_DIR = dirs['models']
FIG_DIR = dirs['figures']


### 1. Load All 4 Experimental Results
Comparing in-dataset performance (Exp 1 & 2) against cross-dataset transfer (Exp 3 & 4).


In [ ]:
import pandas as pd
all_exps = pd.read_csv(TAB_DIR / '05_all_4_experiments_results.csv')
display(all_exps[['experiment', 'model', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc', 'true_phishing_rate', 'predicted_phishing_rate']])


### 2. Summary F1 & ROC-AUC Heatmaps Across Experiments
Visualizing the performance transition from in-dataset to cross-dataset.


In [ ]:
from IPython.display import Image, display

print("F1 Score Heatmap across all 4 experiments:")
display(Image(filename=str(FIG_DIR / 'summary_f1_heatmap.png')))

print("ROC-AUC Score Heatmap across all 4 experiments:")
display(Image(filename=str(FIG_DIR / 'summary_roc_auc_heatmap.png')))


### 3. In-Dataset vs. Cross-Dataset Comparison Bar Chart


In [ ]:
display(Image(filename=str(FIG_DIR / 'in_vs_cross_f1_bar.png')))


### 4. Confusion Matrices for Cross-Dataset Inference
Inspecting the asymmetric prediction errors in Experiment 3 and Experiment 4.


In [ ]:
print("Experiment 3 (PhiUSIIL -> UCI) Confusion Matrices:")
display(Image(filename=str(FIG_DIR / 'cm_Exp3_Logistic_Regression.png')))
display(Image(filename=str(FIG_DIR / 'cm_Exp3_Random_Forest.png')))
display(Image(filename=str(FIG_DIR / 'cm_Exp3_XGBoost.png')))

print("Experiment 4 (UCI -> PhiUSIIL) Confusion Matrices:")
display(Image(filename=str(FIG_DIR / 'cm_Exp4_Logistic_Regression.png')))
display(Image(filename=str(FIG_DIR / 'cm_Exp4_Random_Forest.png')))
display(Image(filename=str(FIG_DIR / 'cm_Exp4_XGBoost.png')))


### 5. Detailed Cross-Dataset Findings
1. **The Generalization Collapse:**
   - In **Exp 3**, models over-predict phishing (predicted rate = 90% vs true 51.6%), leading to high recall (~0.98) but poor precision (~0.56) and accuracy (~59%).
   - In **Exp 4**, models under-predict phishing (predicted rate = 16%–22% vs true 42.6%), leading to severe false negatives and low recall (35%–51%).
2. **Algorithmic Resilience:**
   - **Logistic Regression** is the most resilient model under domain shift ($F_1 = 0.707$ in Exp 3, $F_1 = 0.676$ in Exp 4).
   - High-capacity tree ensembles overfit to non-linear feature quirks that fail out-of-distribution.
